# 第12回：3人ゲームのコアを三角形上に描く

ベンチャー企業の起業ゲームについて，$x_A+x_B+x_C=240$ を満たす配分を正三角形上の点として表す．

- 薄い青：パレート最適な配分全体
- 灰色：個人合理性を満たす配分
- 緑色：すべての提携合理性を満たすコア

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib import font_manager
from matplotlib.patches import Polygon

preferred_fonts = [
    "Hiragino Sans",
    "Yu Gothic",
    "Noto Sans CJK JP",
    "IPAexGothic",
]
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
for font_name in preferred_fonts:
    if font_name in available_fonts:
        plt.rcParams["font.family"] = font_name
        break

plt.rcParams["axes.unicode_minus"] = False


## 配分から平面座標への変換

正三角形の頂点を，Aが全額を受け取る配分，Bが全額を受け取る配分，Cが全額を受け取る配分に対応させる．各配分は3頂点の加重平均として平面上へ写される．

In [ ]:
TOTAL = 240

triangle_vertices = {
    "A": np.array([0.5, np.sqrt(3) / 2]),
    "B": np.array([0.0, 0.0]),
    "C": np.array([1.0, 0.0]),
}


def to_cartesian(allocation, total=TOTAL):
    """配分 (x_A, x_B, x_C) を正三角形上の座標へ変換する．"""
    x_a, x_b, x_c = allocation
    if not np.isclose(x_a + x_b + x_c, total):
        raise ValueError("配分の合計が全員提携の価値と一致しません．")

    return (
        (x_a / total) * triangle_vertices["A"]
        + (x_b / total) * triangle_vertices["B"]
        + (x_c / total) * triangle_vertices["C"]
    )


def polygon_coordinates(allocations):
    return np.array([to_cartesian(allocation) for allocation in allocations])


def constant_coordinate_segment(player, value, total=TOTAL):
    """指定したプレイヤーの利得が一定となる線分の両端を返す．"""
    remainder = total - value
    endpoints = {
        "A": [(value, remainder, 0), (value, 0, remainder)],
        "B": [(remainder, value, 0), (0, value, remainder)],
        "C": [(remainder, 0, value), (0, remainder, value)],
    }[player]
    return polygon_coordinates(endpoints)


## 配分集合とコアの描画

個人合理的な配分は $x_A\geq 60$，$x_B\geq 40$，$x_C\geq 20$ を満たす．コアではさらに $x_A+x_B\geq 200$，$x_A+x_C\geq 150$，$x_B+x_C\geq 100$ を満たす．

In [ ]:
all_allocations = [(240, 0, 0), (0, 240, 0), (0, 0, 240)]
individually_rational = [(180, 40, 20), (60, 160, 20), (60, 40, 140)]
core_vertices = [
    (110, 90, 40),
    (130, 90, 20),
    (140, 80, 20),
    (140, 60, 40),
]

fig, ax = plt.subplots(figsize=(9, 8), constrained_layout=True)

ax.add_patch(
    Polygon(
        polygon_coordinates(all_allocations),
        closed=True,
        facecolor="#D7E8F5",
        edgecolor="#2C5F8A",
        linewidth=2.2,
        label="パレート最適な配分",
    )
)
ax.add_patch(
    Polygon(
        polygon_coordinates(individually_rational),
        closed=True,
        facecolor="#B8B8B8",
        edgecolor="#666666",
        alpha=0.55,
        linewidth=1.8,
        label="個人合理的な配分",
    )
)
ax.add_patch(
    Polygon(
        polygon_coordinates(core_vertices),
        closed=True,
        facecolor="#59A14F",
        edgecolor="#246B24",
        alpha=0.82,
        linewidth=2.2,
        label="コア",
    )
)

individual_guides = [("A", 60), ("B", 40), ("C", 20)]
core_guides = [("A", 140), ("B", 90), ("C", 40)]
guide_colors = {"A": "#A23B3B", "B": "#3B6EA2", "C": "#7651A8"}

for player, value in individual_guides:
    segment = constant_coordinate_segment(player, value)
    ax.plot(
        segment[:, 0],
        segment[:, 1],
        color=guide_colors[player],
        linestyle=":",
        linewidth=1.5,
        alpha=0.65,
        zorder=4,
        label=fr"個人合理性 $x_{player}={value}$",
    )

for player, value in core_guides:
    segment = constant_coordinate_segment(player, value)
    ax.plot(
        segment[:, 0],
        segment[:, 1],
        color=guide_colors[player],
        linestyle=":",
        linewidth=2.5,
        zorder=4,
        label=fr"提携合理性 $x_{player}={value}$",
    )

core_label_offsets = {
    (110, 90, 40): (-105, -20),
    (130, 90, 20): (-100, -6),
    (140, 80, 20): (-100, 14),
    (140, 60, 40): (12, 14),
}

for allocation in core_vertices:
    point = to_cartesian(allocation)
    ax.scatter(*point, color="#174A17", s=36, zorder=5)
    ax.annotate(
        str(allocation),
        point,
        xytext=core_label_offsets[allocation],
        textcoords="offset points",
        fontsize=9,
    )

outside_core = (100, 80, 60)
outside_point = to_cartesian(outside_core)
ax.scatter(
    *outside_point,
    marker="X",
    color="#D62728",
    s=95,
    zorder=6,
    label="コア外の配分 (100, 80, 60)",
)

for player, vertex in triangle_vertices.items():
    offset = {"A": (0, 14), "B": (-46, -18), "C": (8, -18)}[player]
    allocation = {
        "A": "(240, 0, 0)",
        "B": "(0, 240, 0)",
        "C": "(0, 0, 240)",
    }[player]
    ax.annotate(
        f"{player}: {allocation}",
        vertex,
        xytext=offset,
        textcoords="offset points",
        fontsize=10,
        weight="bold",
    )

ax.set_xlim(-0.12, 1.12)
ax.set_ylim(-0.10, 0.99)
ax.set_aspect("equal")
ax.set_title("ベンチャー企業の起業ゲームにおけるコア")
ax.legend(loc="upper right", fontsize=9, framealpha=0.96)
ax.axis("off")

output_path = Path("../figs/12/core_triangle_constraints.png")
output_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output_path, dpi=180, bbox_inches="tight")
plt.show()

print(f"保存先: {output_path}")


## 演習4・課題1の三角形

3人多数決ゲームと3人対称ゲームについて，提携合理性を三角形上に描画する．

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


def unit_coordinates(allocations):
    """合計が1の配分を正三角形上の座標へ変換する．"""
    return np.array(
        [
            x_a * triangle_vertices["A"]
            + x_b * triangle_vertices["B"]
            + x_c * triangle_vertices["C"]
            for x_a, x_b, x_c in allocations
        ]
    )


def unit_constant_segment(player, value):
    remainder = 1 - value
    endpoints = {
        "A": [(value, remainder, 0), (value, 0, remainder)],
        "B": [(remainder, value, 0), (0, value, remainder)],
        "C": [(remainder, 0, value), (0, remainder, value)],
    }[player]
    return unit_coordinates(endpoints)


def draw_unit_simplex(ax, player_labels=("A", "B", "C")):
    simplex = unit_coordinates([(1, 0, 0), (0, 1, 0), (0, 0, 1)])
    ax.add_patch(
        Polygon(
            simplex,
            closed=True,
            facecolor="#E4F0F7",
            edgecolor="#2C5F8A",
            linewidth=2.0,
            zorder=1,
        )
    )
    offsets = [(0, 10), (-18, -16), (8, -16)]
    allocations = ["(1,0,0)", "(0,1,0)", "(0,0,1)"]
    for label, vertex, offset, allocation in zip(
        player_labels, simplex, offsets, allocations
    ):
        ax.annotate(
            f"{label}: {allocation}",
            vertex,
            xytext=offset,
            textcoords="offset points",
            fontsize=9,
            weight="bold",
        )
    ax.set_xlim(-0.10, 1.10)
    ax.set_ylim(-0.10, 0.98)
    ax.set_aspect("equal")
    ax.axis("off")


unit_colors = {"A": "#A23B3B", "B": "#3B6EA2", "C": "#7651A8"}

fig, ax = plt.subplots(figsize=(7.5, 6.5), constrained_layout=True)
draw_unit_simplex(ax)
majority_boundaries = [
    ("C", 0, r"提携 $\{A,B\}$：$x_C=0$"),
    ("B", 0, r"提携 $\{A,C\}$：$x_B=0$"),
    ("A", 0, r"提携 $\{B,C\}$：$x_A=0$"),
]
for player, value, label in majority_boundaries:
    segment = unit_constant_segment(player, value)
    ax.plot(
        segment[:, 0],
        segment[:, 1],
        color=unit_colors[player],
        linestyle=":",
        linewidth=3.2,
        zorder=4,
        label=label,
    )
ax.text(
    0.5,
    0.34,
    "3条件の共通部分なし\nコアは空集合",
    ha="center",
    va="center",
    color="#B22222",
    fontsize=12,
    weight="bold",
    bbox={"facecolor": "white", "edgecolor": "#B22222", "alpha": 0.9},
)
ax.set_title("3人多数決ゲーム：コアが空になる場合")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9)
majority_path = Path("../figs/12/majority_game_empty_core.png")
fig.savefig(majority_path, dpi=180, bbox_inches="tight", facecolor="white")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(13, 4.6), constrained_layout=True)
cases = [
    (0.60, "$a=0.60<2/3$\nコアが存在"),
    (2 / 3, "$a=2/3$\nコアは均等配分のみ"),
    (0.75, "$a=0.75>2/3$\nコアは空集合"),
]
for ax, (a_value, title) in zip(axes, cases):
    draw_unit_simplex(ax, player_labels=("1", "2", "3"))
    upper_bound = 1 - a_value
    for player in ("A", "B", "C"):
        segment = unit_constant_segment(player, upper_bound)
        ax.plot(
            segment[:, 0],
            segment[:, 1],
            color=unit_colors[player],
            linestyle=":",
            linewidth=2.0,
            zorder=3,
        )
    if a_value < 2 / 3:
        low = 1 - 2 * upper_bound
        core = unit_coordinates(
            [(low, upper_bound, upper_bound),
             (upper_bound, low, upper_bound),
             (upper_bound, upper_bound, low)]
        )
        ax.add_patch(
            Polygon(
                core,
                closed=True,
                facecolor="#59A14F",
                edgecolor="#246B24",
                alpha=0.82,
                linewidth=2.0,
                zorder=4,
            )
        )
    elif np.isclose(a_value, 2 / 3):
        center = unit_coordinates([(1 / 3, 1 / 3, 1 / 3)])[0]
        ax.scatter(*center, s=90, color="#246B24", zorder=5)
    else:
        ax.text(
            0.5,
            0.32,
            "共通部分なし",
            ha="center",
            color="#B22222",
            fontsize=11,
            weight="bold",
        )
    ax.set_title(title, fontsize=11)

legend_handles = [
    Line2D([0], [0], color=unit_colors["A"], linestyle=":", linewidth=2, label=r"$x_1=1-a$"),
    Line2D([0], [0], color=unit_colors["B"], linestyle=":", linewidth=2, label=r"$x_2=1-a$"),
    Line2D([0], [0], color=unit_colors["C"], linestyle=":", linewidth=2, label=r"$x_3=1-a$"),
    Patch(facecolor="#59A14F", edgecolor="#246B24", label="コア"),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=4, fontsize=9)
fig.suptitle("3人対称ゲーム：$a$ の値とコアの存在", fontsize=14)
symmetric_path = Path("../figs/12/symmetric_game_core.png")
fig.savefig(symmetric_path, dpi=180, bbox_inches="tight", facecolor="white")
plt.show()

print(f"保存先: {majority_path}")
print(f"保存先: {symmetric_path}")
